# AI Circuit — Kaggle Setup

Supports both workflows:
- **Human+Agent**: you prepare data via `utils/data_prep.py`, agent handles training
- **Full Agent**: fully autonomous — LLM picks classes, preps data, trains

### Prerequisites
- Add the **H&M dataset** to this notebook: *Add Data → H&M Personalized Fashion Recommendations*
- Add your **OpenAI API key** as a Kaggle Secret named `OPENAI_API_KEY`
- Enable **GPU** accelerator in notebook settings (T4 recommended)

In [ ]:
# ── 1. Clone repo and install dependencies ────────────────────────────────
import subprocess, sys

REPO_URL = "https://git.garage.epam.com/ai-circuit/ai-circuit.git"
REPO_DIR = "/kaggle/working/ai_circuit"

subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/ai_circuit

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Setup complete.")

In [ ]:
# ── 2. Set API key from Kaggle Secrets ────────────────────────────────────
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["OPENAI_API_KEY"] = secrets.get_secret("OPENAI_API_KEY")
print("API key loaded.")

In [ ]:
# ── 3. Verify GPU ─────────────────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Workflow A — Human + Agent
You prepare the dataset, agent handles training loop.

In [ ]:
# ── A1. Prepare dataset ───────────────────────────────────────────────────
# Edit max_per_class to control dataset size (500 = fast test, None = full)
import sys
sys.path.insert(0, "/kaggle/working/ai_circuit")

from utils.data_prep import prepare

prepare(
    raw_data_dir="/kaggle/input/h-and-m-personalized-fashion-recommendations",
    output_dir="/kaggle/working/ai_circuit/data/prepared",
    max_per_class=500,   # increase for full run
)

In [ ]:
# ── A2. Patch config for Kaggle and run agent ─────────────────────────────
import yaml

with open("training_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Kaggle-specific overrides
cfg["paths"]["data_dir"]        = "/kaggle/working/ai_circuit/data/prepared"
cfg["paths"]["class_weights"]   = None
cfg["paths"]["class_mapping"]   = None
cfg["training"]["num_workers"]  = 4
cfg["agent"]["experiment_name"] = "kaggle_human_agent"   # edit as needed
cfg["agent"]["max_iterations"]  = 3                      # edit as needed

from agents.training_agent import run as run_training
run_training(config_path=cfg, max_iterations=cfg["agent"]["max_iterations"],
             target_f1=cfg["agent"]["target_f1"], workflow="human+agent")

---
## Workflow B — Full Agent
Fully autonomous — LLM selects classes, prepares data, trains.

In [ ]:
# ── B1. Patch config and run full agent ───────────────────────────────────
import yaml, sys
sys.path.insert(0, "/kaggle/working/ai_circuit")

with open("training_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Kaggle-specific overrides
cfg["data_prep"]["raw_data_dir"]       = "/kaggle/input/h-and-m-personalized-fashion-recommendations"
cfg["data_prep"]["max_train_per_class"] = 500   # increase for full run
cfg["training"]["num_workers"]          = 4
cfg["agent"]["experiment_name"]         = "kaggle_full_agent"   # edit as needed
cfg["agent"]["max_iterations"]          = 3

from agents.data_prep_agent import prepare_data
from agents.training_agent import run as run_training

exp_name = cfg["agent"].get("experiment_name") or "auto"
data_prep_cfg = cfg["data_prep"]

data_paths = prepare_data(
    raw_data_dir=data_prep_cfg["raw_data_dir"],
    output_dir=f"/kaggle/working/ai_circuit/data/{exp_name.replace(' ', '_')}",
    max_train_per_class=data_prep_cfg.get("max_train_per_class"),
    llm_model=cfg["agent"]["llm_model"],
    instructions=data_prep_cfg.get("instructions"),
    force_classes=data_prep_cfg.get("force_classes"),
)

cfg["paths"]["data_dir"]      = data_paths["data_dir"]
cfg["paths"]["class_weights"] = None
cfg["paths"]["class_mapping"] = None

run_training(config_path=cfg, max_iterations=cfg["agent"]["max_iterations"],
             target_f1=cfg["agent"]["target_f1"], workflow="full_agent")

---
## Results
Experiment logs and checkpoints saved under `/kaggle/working/ai_circuit/experiments/`

In [ ]:
# ── View experiment log ───────────────────────────────────────────────────
import json, glob

logs = sorted(glob.glob("/kaggle/working/ai_circuit/experiments/*/experiment_log.json"))
for log_path in logs:
    print(f"\n=== {log_path} ===")
    with open(log_path) as f:
        for run in json.load(f):
            print(f"  Run {run['run']} | backbone={run['backbone']} | macro_f1={run['macro_f1']:.4f} | epochs={run['epochs_trained']}")